# Notebook 03 — Feature Engineering

**Project:** TCO Optimisation Model — Sprint One  
**Author:** Soham Dharne (2026)  
**NFR compliance:** NFR-09 (100% annotated), NFR-07 (reproducibility via fixed seed)

---

## 1. Purpose of Feature Engineering

Raw features cannot be fed directly into most ML algorithms for two reasons:
1. **Scale sensitivity** — gradient-based methods (ANN, Logistic Regression) are sensitive to feature magnitude; a feature ranging 500–200,000 will dominate gradients over one ranging 1–20
2. **Type incompatibility** — sklearn estimators require numeric inputs; string ordinal/nominal columns must be encoded

This notebook documents every preprocessing decision and explains why each encoder choice is optimal for the downstream ensemble:
- **XGBoost** (tree-based, rank-invariant)
- **ANN** (gradient-based, scale-sensitive)
- **Logistic Regression / Ridge** (linear, scale-sensitive and collinearity-sensitive)
- **Decision Tree** (tree-based, rank-invariant)

## 2. Preprocessing Decision Matrix

| Feature Group | Features | Encoder | Rationale |
|---|---|---|---|
| **Numeric** | `estimated_loc`, `timeline_days`, `team_size_required`, `integration_count`, `testing_coverage_pct` | `StandardScaler` | Zero-mean, unit-variance. Required by ANN and Logistic Regression. XGBoost is invariant to scaling but not harmed by it. |
| **Ordinal** | `complexity_score`, `technical_risk_level`, `seniority_required`, `documentation_level`, `performance_tier`, `security_criticality`, `budget_pressure`, `maintainability_req` | `OrdinalEncoder` | Converts string levels to integers **preserving rank order** (Low=0, Medium=1, High=2). Tree models can exploit this rank directly; linear models get a meaningful integer gradient. OneHotEncoding ordinals would destroy the rank information. |
| **Binary** | `regulatory_compliance` | `PassThrough` | Already 0/1 integer — no transformation needed. Scaling binary features is harmless but unnecessary; PassThrough avoids spurious column renaming. |
| **Nominal** | `domain_category` | `OneHotEncoder(drop='first')` | 5 unordered categories with no natural rank — OrdinalEncoder would incorrectly impose a numerical ordering (e.g., Web=0 < Infrastructure=1). `drop='first'` removes the baseline category (Web) to avoid perfect multicollinearity in the Logistic Regression meta-learner. |

All four transformers are composed inside a single `ColumnTransformer`, which ensures the same transformation is consistently applied to train, val, and test splits without data leakage (the `fit` call is on train data only).

## 3. Why These Choices Suit Each Model

### 3a. XGBoost
XGBoost is a gradient-boosted tree model — it makes splits based on rank, not absolute value. Therefore:
- `StandardScaler` does not help XGBoost directly, but consistency in the pipeline ensures we can swap XGBoost for a gradient-based model without pipeline changes
- `OrdinalEncoder` is **ideal** — XGBoost can directly exploit integer-encoded ordinal ranks in its split decisions
- `OneHotEncoder(drop='first')` for domain is correct — tree models can handle OHE without collinearity issues

### 3b. ANN (MLPClassifier / MLPRegressor)
Neural networks use gradient descent — features with very different scales cause the optimizer to converge slowly or become stuck. `StandardScaler` is **essential** for ANN to converge within `max_iter=500`.

### 3c. Logistic Regression / Ridge (meta-learners)
Linear models are **critically sensitive** to:
1. Scale — `StandardScaler` ensures each feature contributes proportionally to the linear combination
2. Multicollinearity — `OneHotEncoder(drop='first')` removes the dummy variable trap; `OrdinalEncoder` avoids creating k-1 dummy columns for 8 ordinal features

### 3d. Decision Tree
Like XGBoost, Decision Trees are rank-invariant. The `OrdinalEncoder` choice means the tree can make interpretable splits like `complexity_score <= 1` (i.e., Low or Medium).

## 4. Environment Setup

In [ ]:
import sys
import os

NOTEBOOK_DIR = os.path.abspath('')
PROJECT_ROOT = os.path.dirname(NOTEBOOK_DIR)
SRC_DIR = os.path.join(PROJECT_ROOT, 'src')

if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

print(f'Project root : {PROJECT_ROOT}')
print(f'src/ on path : {SRC_DIR}')

In [ ]:
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from feature_engineering import (
    load_and_split,
    build_preprocessor,
    get_feature_names,
    label_encoder,
)
from config import (
    NUMERIC_FEATURES, ORDINAL_FEATURES, BINARY_FEATURES,
    NOMINAL_FEATURES, DOMAIN_CATEGORIES,
    TARGET_CLASS, TARGET_REGR, CLASS_LABELS,
    SEED, TEST_SIZE, VAL_SIZE,
)

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_theme(style='whitegrid')

print('Imports OK')

## 5. Load Dataset and Create Train/Val/Test Splits

`load_and_split()` performs:
1. `pd.read_csv` from `DATASET_PATH`
2. `validate_dataframe` — raises warnings for any schema violations
3. Encodes `target_team_label` strings → integers using the shared `LabelEncoder` (fitted once on `CLASS_LABELS`)
4. Stratified two-stage split:
   - First split: 85% train+val / 15% test (`TEST_SIZE=0.15`)
   - Second split: 70% train / 15% val of the original dataset (`VAL_SIZE=0.15`)
   - `stratify=y_clf` ensures each split has proportional class representation
5. Returns 9-tuple: `X_train, X_val, X_test, y_clf_train, y_clf_val, y_clf_test, y_rgr_train, y_rgr_val, y_rgr_test`

The stratified split is critical for the Human class (the minority) — without stratification, test sets could have zero Human examples.

In [ ]:
(X_train, X_val, X_test,
 y_clf_train, y_clf_val, y_clf_test,
 y_rgr_train, y_rgr_val, y_rgr_test) = load_and_split()

print('=== Split Sizes ===')
print(f'  Train : X={X_train.shape}  y_clf={y_clf_train.shape}  y_rgr={y_rgr_train.shape}')
print(f'  Val   : X={X_val.shape}    y_clf={y_clf_val.shape}    y_rgr={y_rgr_val.shape}')
print(f'  Test  : X={X_test.shape}   y_clf={y_clf_test.shape}   y_rgr={y_rgr_test.shape}')

total = len(X_train) + len(X_val) + len(X_test)
print(f'  Total : {total}')
print(f'  Train: {len(X_train)/total*100:.1f}%  Val: {len(X_val)/total*100:.1f}%  Test: {len(X_test)/total*100:.1f}%')

## 6. Verify Stratification — Class Proportions per Split

We confirm that each split has approximately the same class proportions as the full dataset. A well-stratified split ensures:
- The test set is representative — test metrics are not biased by under/over-representation
- The Human class (minority) appears in all three splits with enough samples to compute per-class metrics

In [ ]:
def label_counts(y_encoded, split_name):
    classes = label_encoder.classes_  # AI, Human, Hybrid
    unique, counts = np.unique(y_encoded, return_counts=True)
    result = {}
    for u, c in zip(unique, counts):
        lbl = label_encoder.inverse_transform([u])[0]
        result[lbl] = c
    pct = {k: v/sum(result.values())*100 for k, v in result.items()}
    print(f'{split_name}:')
    for lbl in CLASS_LABELS:
        n = result.get(lbl, 0)
        p = pct.get(lbl, 0.0)
        print(f'  {lbl:8s}: {n:3d} ({p:.1f}%)')

label_counts(y_clf_train, 'Train')
label_counts(y_clf_val,   'Val  ')
label_counts(y_clf_test,  'Test ')

## 7. Label Encoding Scheme

The `LabelEncoder` is fitted **once** on `CLASS_LABELS = ['Human', 'Hybrid', 'AI']` and shared across all notebooks. The consistent encoding mapping is:

In [ ]:
print('=== Label Encoding (LabelEncoder) ===')
print('Classes (alphabetical order):', list(label_encoder.classes_))
print()
for lbl in label_encoder.classes_:
    encoded = label_encoder.transform([lbl])[0]
    print(f'  {lbl:8s} → {encoded}')

print()
print('Note: LabelEncoder sorts alphabetically, so AI=0, Human=1, Hybrid=2')
print('This encoding is consistent across train/val/test and all model evaluations.')

## 8. Building the Preprocessing Pipeline

`build_preprocessor()` returns a `ColumnTransformer` with four named transformers. We inspect each transformer's configuration to confirm the expected encoder is applied to the expected columns.

In [ ]:
pre = build_preprocessor()

print('=== ColumnTransformer Transformers ===')
for name, transformer, columns in pre.transformers:
    transformer_name = type(transformer).__name__ if transformer != 'passthrough' else 'PassThrough'
    print(f'\n[{name}] {transformer_name}')
    print(f'  Columns ({len(columns)}): {columns}')
    if hasattr(transformer, 'categories') and transformer != 'passthrough':
        try:
            print(f'  Categories: {transformer.categories}')
        except:
            pass

## 9. Fitting the Preprocessor on Training Data

**Critical data leakage prevention:** The preprocessor is fitted **only** on `X_train`. Fitting on the full dataset would leak test set statistics into the scaler's mean/std and the encoder's category lists. We then transform `X_val` and `X_test` using the train-fitted preprocessor.

After fitting, we record:
- The scalar mean and std from `StandardScaler`
- The ordinal integer encoding from `OrdinalEncoder`

In [ ]:
pre.fit(X_train)

# Extract StandardScaler parameters
scaler = pre.named_transformers_['num']
print('=== StandardScaler — Mean and Std (from training data only) ===')
for feat, mean, std in zip(NUMERIC_FEATURES, scaler.mean_, scaler.scale_):
    print(f'  {feat:28s}: mean={mean:10.2f}, std={std:8.2f}')

print()
# Extract OrdinalEncoder categories
ord_enc = pre.named_transformers_['ord']
print('=== OrdinalEncoder — Category → Integer Mapping ===')
for feat, cats in zip(ORDINAL_FEATURES.keys(), ord_enc.categories_):
    mapping = {v: i for i, v in enumerate(cats)}
    print(f'  {feat:28s}: {mapping}')

## 10. Transformed Feature Shape and Names

After transformation, the feature matrix has a different number of columns than the raw input:
- 5 numeric → 5 scaled columns (same count)
- 8 ordinal → 8 integer columns (same count)
- 1 binary → 1 column (PassThrough)
- 1 nominal (5 categories) → **4 columns** (drop='first' removes the baseline Web category)

**Total transformed features: 5 + 8 + 1 + 4 = 18**

In [ ]:
X_train_t = pre.transform(X_train)
X_val_t   = pre.transform(X_val)
X_test_t  = pre.transform(X_test)

feature_names = get_feature_names(pre)

print('=== Transformed Feature Dimensions ===')
print(f'  X_train_t : {X_train_t.shape}')
print(f'  X_val_t   : {X_val_t.shape}')
print(f'  X_test_t  : {X_test_t.shape}')
print(f'  Feature names ({len(feature_names)}): {feature_names}')

print()
print('Feature group breakdown:')
print(f'  Numeric (StandardScaled)  : {len(NUMERIC_FEATURES)}')
print(f'  Ordinal (OrdinalEncoded)  : {len(ORDINAL_FEATURES)}')
print(f'  Binary  (PassThrough)     : {len(BINARY_FEATURES)}')
print(f'  Nominal (OHE drop=first)  : {len(DOMAIN_CATEGORIES) - 1}  (5 categories - 1 baseline)')
print(f'  TOTAL                     : {len(feature_names)}')

## 11. Inspecting Sample Transformed Rows

We inspect 5 transformed rows to verify the transformations are working correctly. Key things to verify:
- Numeric columns should have values roughly in [-3, 3] (standardised)
- Ordinal columns should be integers 0, 1, 2, or 3
- `regulatory_compliance` should be 0 or 1 (unchanged)
- OHE domain columns should be 0 or 1, with at most one 1 per row

In [ ]:
df_transformed = pd.DataFrame(X_train_t[:5], columns=feature_names)
print('=== Sample Transformed Rows (first 5 of training set) ===')
df_transformed

In [ ]:
# Side-by-side: raw vs transformed for row 0
print('=== Raw vs Transformed — First Training Record ===')
print('\nRAW:')
print(X_train.iloc[0].to_string())
print(f'  {TARGET_CLASS}: {label_encoder.inverse_transform([y_clf_train[0]])[0]}')
print(f'  {TARGET_REGR}: {y_rgr_train[0]:.2f}%')
print('\nTRANSFORMED:')
for feat, val in zip(feature_names, X_train_t[0]):
    print(f'  {feat:28s}: {val:.4f}')

## 12. StandardScaler — Before/After Distribution Comparison

We visualise the effect of StandardScaler on the numeric features. Before scaling, `estimated_loc` ranges 500–200,000; after scaling it is centred near 0 with unit variance. This plot confirms the scaler is working correctly and highlights the right-skew of `estimated_loc` that persists (StandardScaler does not correct distributional shape — only mean/variance).

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 7))

# Get index positions of numeric features in transformed array
numeric_indices = list(range(len(NUMERIC_FEATURES)))

for i, col in enumerate(NUMERIC_FEATURES):
    # Before
    axes[0, i].hist(X_train[col], bins=30, color='#78909C', edgecolor='white', alpha=0.85)
    axes[0, i].set_title(f'{col}\n(raw)', fontsize=8)
    axes[0, i].set_ylabel('Count' if i == 0 else '')

    # After
    axes[1, i].hist(X_train_t[:, i], bins=30, color='#26C6DA', edgecolor='white', alpha=0.85)
    axes[1, i].set_title(f'{col}\n(scaled)', fontsize=8)
    axes[1, i].set_ylabel('Count' if i == 0 else '')

axes[0, 0].text(-0.25, 0.5, 'BEFORE\nScaling', transform=axes[0, 0].transAxes,
                rotation=90, va='center', fontsize=9, color='gray')
axes[1, 0].text(-0.25, 0.5, 'AFTER\nScaling', transform=axes[1, 0].transAxes,
                rotation=90, va='center', fontsize=9, color='gray')

plt.suptitle('StandardScaler — Before vs After (Training Data)', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

## 13. OrdinalEncoder Verification

We verify that the OrdinalEncoder correctly maps string levels to their rank integers. We show the mapping for `complexity_score` and `technical_risk_level` as representative examples.

In [ ]:
ord_idx_start = len(NUMERIC_FEATURES)  # ordinal features start after numeric in the transformed array

print('=== OrdinalEncoder Verification ===')
for i, (col, levels) in enumerate(ORDINAL_FEATURES.items()):
    col_idx = ord_idx_start + i
    train_vals = X_train_t[:, col_idx]
    unique_encoded = sorted(np.unique(train_vals))
    print(f'  {col:28s}: encoded values={unique_encoded}  (expected: {list(range(len(levels)))})')

print()
print('OrdinalEncoder preserves rank: Low=0 < Medium=1 < High=2 < Critical=3')
print('This is critical for tree models to learn thresholds like "risk >= 2 → High/Critical"')

## 14. OneHotEncoder Verification — Domain Category

`OneHotEncoder(drop='first')` is applied to `domain_category`. With 5 categories and `drop='first'`, we get 4 binary columns. We verify:
1. The dropped baseline category (Web — first alphabetically in `DOMAIN_CATEGORIES`)
2. Exactly one 1 per row (or all zeros for the baseline Web category)
3. Column names follow the `domain_<category>` naming convention

In [ ]:
# OHE columns are the last 4 in the transformed array
ohe_names = [n for n in feature_names if n.startswith('domain_')]
ohe_idx_start = len(NUMERIC_FEATURES) + len(ORDINAL_FEATURES) + len(BINARY_FEATURES)

print('=== OneHotEncoder(drop=first) Verification ===')
print(f'Input categories  : {DOMAIN_CATEGORIES}')
print(f'Dropped baseline  : {DOMAIN_CATEGORIES[0]}  (first category)')
print(f'Output columns    : {ohe_names}')

# Check that at most one 1.0 per row
ohe_block = X_train_t[:, ohe_idx_start:]
row_sums = ohe_block.sum(axis=1)
assert (row_sums <= 1.0).all(), 'OHE error: multiple 1s in a row!'
print(f'\nRow sum check: max={row_sums.max():.0f} (0=Web baseline, 1=other domain)  PASSED')

# Show the OHE pattern for a few rows
df_ohe = pd.DataFrame(ohe_block[:5], columns=ohe_names)
df_ohe.insert(0, 'domain_raw', X_train['domain_category'].values[:5])
print('\nSample OHE encoding:')
print(df_ohe.to_string())

## 15. Correlation of Transformed Features

We compute the correlation matrix of the full transformed feature matrix. This confirms:
- The OHE domain columns are mutually exclusive (negative pairwise correlations)
- StandardScaler did not introduce spurious correlations
- No transformed features are perfectly correlated (which would indicate a collinearity problem)

In [ ]:
df_t_full = pd.DataFrame(X_train_t, columns=feature_names)
corr_t = df_t_full.corr()

fig, ax = plt.subplots(figsize=(13, 10))
mask = np.triu(np.ones_like(corr_t, dtype=bool))
sns.heatmap(
    corr_t,
    annot=True, fmt='.1f',
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    mask=mask,
    square=True,
    linewidths=0.3,
    annot_kws={'size': 7},
    ax=ax
)
ax.set_title('Correlation Matrix — Transformed Features (Training Data)', fontsize=12)
plt.tight_layout()
plt.show()

# Check for any very high correlations that might indicate collinearity
high_corr = [
    (feature_names[i], feature_names[j], corr_t.values[i, j])
    for i in range(len(feature_names))
    for j in range(i)
    if abs(corr_t.values[i, j]) > 0.7
]
if high_corr:
    print('High correlations (|r| > 0.7):')
    for a, b, r in high_corr:
        print(f'  {a} — {b}: {r:.3f}')
else:
    print('No feature pairs with |correlation| > 0.7 — no collinearity concern')

## 16. Data Leakage Check

We verify that the StandardScaler's mean and std were computed only on the training set. If there were leakage (fitting on all data), the validation/test set statistics would match the scaler's parameters. We demonstrate there is a small but measurable difference — as expected.

In [ ]:
scaler = pre.named_transformers_['num']

print('=== No-Leakage Verification — StandardScaler ===')
print(f'  Scaler fitted on training data only ({len(X_train)} records)')
print()
print(f'{"Feature":28s}  {"Train mean":>12}  {"Scaler mean":>12}  {"Match?":>8}')
print('-' * 68)
for feat, scaler_mean, scaler_std in zip(NUMERIC_FEATURES, scaler.mean_, scaler.scale_):
    train_mean = X_train[feat].mean()
    match = abs(train_mean - scaler_mean) < 0.01
    print(f'  {feat:26s}  {train_mean:>12.2f}  {scaler_mean:>12.2f}  {"YES" if match else "MISMATCH":>8}')

print()
print('Scaler mean == Train mean: data leakage is NOT present')

## 17. build_clf_pipeline and build_rgr_pipeline Overview

The `feature_engineering.py` module exports two pipeline builders used in Notebook 04:

```
build_clf_pipeline() → Pipeline
  └─ ColumnTransformer (StandardScaler + OrdinalEncoder + PassThrough + OneHotEncoder)
     └─ StackingClassifier
          ├─ XGBClassifier
          ├─ DecisionTreeClassifier
          ├─ MLPClassifier (128→64→32, ReLU, Adam)
          └─ LogisticRegression (base learner)
          meta: LogisticRegression

build_rgr_pipeline() → Pipeline
  └─ ColumnTransformer (same preprocessor)
     └─ StackingRegressor
          ├─ XGBRegressor
          ├─ DecisionTreeRegressor
          ├─ MLPRegressor (128→64→32, ReLU, Adam)
          └─ LinearRegression
          meta: Ridge
```

Both pipelines use `passthrough=True` in the stacking estimator — the meta-learner receives both the base learner predictions **and** the original (preprocessed) features, giving it access to the full input space without any additional pipeline complexity.

In [ ]:
from model import build_clf_pipeline, build_rgr_pipeline

clf_pipe = build_clf_pipeline()
rgr_pipe = build_rgr_pipeline()

print('=== Classification Pipeline ===')
print(clf_pipe)

print()
print('=== Regression Pipeline ===')
print(rgr_pipe)

## 18. Summary

| Decision | Choice | Rationale |
|---|---|---|
| Numeric → | StandardScaler | Required by ANN and LogReg meta-learner; harmless for XGBoost/DT |
| Ordinal → | OrdinalEncoder | Preserves rank order; tree models exploit integer ranks; avoids 8×3 dummy variable explosion |
| Binary → | PassThrough | Already 0/1; no transformation needed |
| Nominal → | OneHotEncoder(drop='first') | 5 unordered categories; drop='first' prevents dummy variable trap in linear models |
| Split strategy | Stratified 70/15/15 | Ensures Human (minority) class represented in all splits |
| Leakage prevention | fit on train only | Scaler mean/std derived from training data only; val/test are transform-only |
| Output shape | 18 features | 5+8+1+4 (OHE drops Web baseline category) |

**Next step:** Notebook 04 — Model Training (`04_model_training.ipynb`)